In [1]:
from make_dataset import MultiMatchSoccerDataset, organize_and_process
from torch.utils.data import DataLoader, Subset
from utils.utils import set_evertyhing, worker_init_fn, generator
from utils.data_utils import split_dataset_indices, custom_collate_fn
import torch
import os


# 1. Hyperparameter Setting
raw_data_path = "idsse-data"
data_save_path = "match_data"
batch_size = 64
num_workers = 8
epochs = 100
learning_rate = 1e-4
SEED = 42

set_evertyhing(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2. Data Loading
print("---Data Loading---")
if not os.path.exists(data_save_path) or len(os.listdir(data_save_path)) == 0:
    organize_and_process(raw_data_path, data_save_path)
else:
    print("Skip organize_and_process")

dataset = MultiMatchSoccerDataset(data_root=data_save_path, use_condition_graph=True)
train_idx, test_idx, _, _ = split_dataset_indices(dataset, random_seed=SEED)

train_dataloader = DataLoader(
    Subset(dataset, train_idx),
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=True,
    persistent_workers=False,
    collate_fn=custom_collate_fn,
    worker_init_fn=worker_init_fn,
    generator=generator(SEED)
)

test_dataloader = DataLoader(
    Subset(dataset, test_idx),
    batch_size=8,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
    persistent_workers=False,
    collate_fn=custom_collate_fn,
    worker_init_fn=worker_init_fn
)

---Data Loading---
Skip organize_and_process


Data Loading...: 100%|██████████| 6/6 [02:21<00:00, 23.58s/it]


In [2]:
sample = dataset[0]

In [3]:
sample['condition_graph_seq']

[HeteroData(
   Attk={ x=[11, 7] },
   Def={ x=[11, 7] },
   Ball={ x=[1, 4] },
   (Attk, interaction, Attk)={
     edge_index=[2, 121],
     edge_attr=[121, 1],
   },
   (Attk, interaction, Def)={
     edge_index=[2, 121],
     edge_attr=[121, 1],
   },
   (Def, interaction, Def)={
     edge_index=[2, 121],
     edge_attr=[121, 1],
   },
   (Attk, interaction, Ball)={
     edge_index=[2, 11],
     edge_attr=[11, 1],
   },
   (Def, interaction, Ball)={
     edge_index=[2, 11],
     edge_attr=[11, 1],
   }
 ),
 HeteroData(
   Attk={ x=[11, 7] },
   Def={ x=[11, 7] },
   Ball={ x=[1, 4] },
   (Attk, interaction, Attk)={
     edge_index=[2, 121],
     edge_attr=[121, 1],
   },
   (Attk, interaction, Def)={
     edge_index=[2, 121],
     edge_attr=[121, 1],
   },
   (Def, interaction, Def)={
     edge_index=[2, 121],
     edge_attr=[121, 1],
   },
   (Attk, interaction, Ball)={
     edge_index=[2, 11],
     edge_attr=[11, 1],
   },
   (Def, interaction, Ball)={
     edge_index=[2, 11],
   

In [4]:
data = sample['condition_graph_seq'][0]

In [5]:
data['Attk', 'interaction', 'Def'].edge_attr

tensor([[1.3376],
        [0.7679],
        [1.3818],
        [1.1368],
        [1.3298],
        [1.3792],
        [0.8223],
        [0.9906],
        [0.4866],
        [1.5015],
        [0.6719],
        [1.0293],
        [0.5886],
        [0.1137],
        [0.1548],
        [0.2818],
        [1.6870],
        [1.3725],
        [0.3991],
        [0.8562],
        [0.6379],
        [0.6665],
        [1.2551],
        [0.0712],
        [0.6607],
        [0.4573],
        [0.5453],
        [1.6982],
        [1.2274],
        [0.5514],
        [0.5506],
        [1.0963],
        [0.4722],
        [0.5657],
        [0.6408],
        [0.6405],
        [0.4692],
        [0.7689],
        [1.1603],
        [0.8654],
        [0.1688],
        [0.5351],
        [0.4820],
        [0.3654],
        [0.9425],
        [0.4576],
        [0.9508],
        [0.7047],
        [0.9485],
        [1.2150],
        [0.7113],
        [0.5155],
        [0.0321],
        [1.0284],
        [0.1954],
        [0

In [ ]:
sample["condition_columns"]

['Away_22_x',
 'Away_22_y',
 'Away_25_x',
 'Away_25_y',
 'Away_26_x',
 'Away_26_y',
 'Away_29_x',
 'Away_29_y',
 'Away_31_x',
 'Away_31_y',
 'Away_33_x',
 'Away_33_y',
 'Away_34_x',
 'Away_34_y',
 'Away_35_x',
 'Away_35_y',
 'Away_36_x',
 'Away_36_y',
 'Away_37_x',
 'Away_37_y',
 'Away_39_x',
 'Away_39_y']

In [9]:
cond_names = dataset[0]["condition_columns"]
target_names = dataset[0]["target_columns"]

# 수비수 이름 순서 (condition)
cond_defenders = [name.rsplit('_', 1)[0] for name in cond_names if name.startswith("Away_") and name.endswith("_x")]
# 타깃 이름 순서 (target)
target_defenders = [name.rsplit('_', 1)[0] for name in target_names[::2]]  # x만 가져오기

print("Condition 수비수 순서:", cond_defenders)
print("Target 수비수 순서:   ", target_defenders)

if cond_defenders != target_defenders:
    print("❗️ 수비수 순서가 condition과 target 간에 다릅니다!")
else:
    print("✅ 수비수 순서가 일치합니다.")


Condition 수비수 순서: ['Away_22', 'Away_25', 'Away_26', 'Away_29', 'Away_31', 'Away_33', 'Away_34', 'Away_35', 'Away_36', 'Away_37', 'Away_39']
Target 수비수 순서:    ['Away_22', 'Away_25', 'Away_26', 'Away_29', 'Away_31', 'Away_33', 'Away_34', 'Away_35', 'Away_36', 'Away_37', 'Away_39']
✅ 수비수 순서가 일치합니다.


In [27]:
def extract_defender_xy_from_condition(sample):
    condition = sample["condition"]  # [T, F]
    columns = sample["condition_columns"]

    # 수비수 부분만 슬라이싱 (공격수 제외, 볼 제외)
    defender_cols = columns[11*7 : -4]  # ['Away_22_x', ..., 'Away_39_starter']

    # defender feature 수: 11명 * 7 features
    assert len(defender_cols) == 11 * 7

    # x/y 인덱스만 뽑기
    x_indices = [11*7 + i for i, col in enumerate(defender_cols) if col.endswith('_x')]
    y_indices = [11*7 + i for i, col in enumerate(defender_cols) if col.endswith('_y')]

    x_vals = sample["condition"][-1, x_indices]  # [11]
    y_vals = sample["condition"][-1, y_indices]  # [11]

    return torch.stack([x_vals, y_vals], dim=-1)  # [11, 2]


In [32]:
cond_def_last = extract_defender_xy_from_condition(sample)  # [11, 2]
target_start = sample["target"][0].reshape(11, 2)  # [11, 2]

diff = torch.norm(cond_def_last - target_start, dim=-1)
print("수비수 시작 위치 차이 (평균):", diff.mean().item())


수비수 시작 위치 차이 (평균): 0.0034273953642696142


In [29]:
cond_def_last

tensor([[-0.6042,  0.0547],
        [ 0.4381, -0.3462],
        [-0.1909, -0.5079],
        [ 0.3893,  0.3597],
        [-0.2333,  0.0635],
        [-0.0623,  0.6956],
        [-0.0008, -0.0182],
        [-0.1771,  0.4888],
        [-0.2126, -0.2100],
        [ 0.0667,  0.2871],
        [ 0.0192, -0.3718]])

torch.Size([22])